[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/18_batchnorm.ipynb)

# 🟡 Medium: BatchNorm with Running Stats (nnx.Module)

*Core Ops & Layers*
Implement **Batch Normalization** with running statistics.

**Training** — normalise with the statistics of the current batch, and update
the running buffers:

$$\mu_{run} \leftarrow m\,\mu_{run} + (1-m)\,\mu_{batch}$$

**Inference** — normalise with the stored running statistics, update nothing.

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}\gamma + \beta$$

### Rules
- Signature: `BatchNorm(num_features, *, momentum=0.9, eps=1e-5, rngs=None)`
- `__call__(x, use_running_average=False)`
- `self.scale`, `self.bias` → `nnx.Param` (ones / zeros)
- `self.running_mean`, `self.running_var` → `nnx.BatchStat` (zeros / ones)
- Reduce over **all axes except the last** — works for `(N, C)` and `(N, H, W, C)`
- Use the **biased** batch variance
- Running buffers must update **only** in training mode

### Why `nnx.BatchStat` and not `nnx.Param`
Running statistics are **state**, not parameters: they are updated by an
exponential moving average, not by gradient descent. Tagging them `BatchStat`
lets you filter them out when you build the optimizer:

```python
params = nnx.state(model, nnx.Param)        # optimizer sees only these
stats  = nnx.state(model, nnx.BatchStat)    # carried along, never differentiated
```

NNX modules are **mutable**, so `self.running_mean[...] = ...` inside `__call__`
just works — no threading of a `mutable` collection through every call the way
Linen requires. This is the clearest demonstration of what NNX buys you.

### The classic gotcha
Forgetting to switch to eval mode means inference normalises by whatever happens
to be in the current batch, so predictions change depending on what else you
batched alongside them — and with batch size 1 the variance is 0 and everything
collapses.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class BatchNorm(nnx.Module):
    """BatchNorm over the last (feature) axis, with running statistics."""

    def __init__(self, num_features: int, *, momentum: float = 0.9,
                 eps: float = 1e-5, rngs: nnx.Rngs = None):
        pass  # Replace this

    def __call__(self, x, use_running_average: bool = False):
        """(..., num_features) -> same shape."""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

bn = BatchNorm(4)
x = jax.random.normal(jax.random.key(0), (32, 4)) * 3.0 + 5.0

print("running_mean before:", bn.running_mean[...])
out = bn(x)                       # training mode
print("running_mean after: ", bn.running_mean[...])
print("train-mode output mean:", out.mean(0), "(~0)")

eval_out = bn(x, use_running_average=True)
print("eval-mode output mean: ", eval_out.mean(0), "(NOT ~0 — uses running stats)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("batchnorm")

# hint("batchnorm")      # stuck? nudge without the answer
# solution("batchnorm")  # spoiler: the reference implementation